# Jour 4 — Tokens, BPE et fenêtre de contexte

Notebook généré à partir du Markdown source.

## Estimation simple de tokens

Cette estimation est pédagogique. En production, utiliser le tokenizer réel du modèle.

In [ ]:
from math import ceil

def estimate_tokens(text: str, chars_per_token: float = 4.0) -> int:
    if chars_per_token <= 0:
        raise ValueError('chars_per_token must be positive')
    return ceil(len(text) / chars_per_token)

estimate_tokens('AI Engineering Bootcamp')


## Mini BPE

On entraîne un tokenizer BPE pédagogique sur un corpus minimal.

In [ ]:
from collections import Counter

END_WORD = '</w>'

def word_to_symbols(word):
    return tuple(word) + (END_WORD,)

def corpus_to_vocab(corpus):
    counts = Counter(corpus.lower().split())
    return {word_to_symbols(word): freq for word, freq in counts.items()}

def get_pair_stats(vocab):
    pairs = Counter()
    for symbols, freq in vocab.items():
        for i in range(len(symbols) - 1):
            pairs[(symbols[i], symbols[i + 1])] += freq
    return pairs

corpus = 'ai engineering ai engineer engineering systems context windows tokens tokens'
vocab = corpus_to_vocab(corpus)
get_pair_stats(vocab).most_common(10)


In [ ]:
def merge_pair_in_symbols(symbols, pair, replacement):
    result = []
    i = 0
    while i < len(symbols):
        if i < len(symbols) - 1 and (symbols[i], symbols[i + 1]) == pair:
            result.append(replacement)
            i += 2
        else:
            result.append(symbols[i])
            i += 1
    return tuple(result)

merge_pair_in_symbols(tuple('low'), ('l', 'o'), 'lo')


## Budget de contexte

Le budget total doit réserver de la place pour la sortie.

In [ ]:
def build_context_plan(context_window, reserved_output_tokens, sections, chars_per_token=4.0):
    section_tokens = {name: estimate_tokens(value, chars_per_token) for name, value in sections.items()}
    estimated_input_tokens = sum(section_tokens.values())
    remaining_tokens = context_window - reserved_output_tokens - estimated_input_tokens
    return {
        'context_window': context_window,
        'reserved_output_tokens': reserved_output_tokens,
        'estimated_input_tokens': estimated_input_tokens,
        'remaining_tokens': remaining_tokens,
        'fits': remaining_tokens >= 0,
        'sections': section_tokens,
    }

build_context_plan(16000, 1500, {
    'system': 'You are a precise assistant.',
    'history': 'Previous exchange. ' * 300,
    'documents': 'Technical documentation. ' * 900,
    'question': 'Explain token budgeting.'
})
